# Practice Notebook: Transformers & Hugging Face (Beginner)
This notebook provides **hands-on practice** with Hugging Face Transformers:

1. Sentiment pipeline (no training)
2. Tokenization (input_ids, attention_mask)
3. BERT embeddings / hidden states
4. Zero-shot classification
5. Fine-tuning DistilBERT on IMDb (small subset)
6. Testing the fine-tuned model

> Designed for **Google Colab** (run cell-by-cell).

## Setup: Install Libraries

In [ ]:
!pip -q install transformers datasets evaluate accelerate
import numpy as np
import torch
import matplotlib.pyplot as plt
print('Setup complete')

## Practice 1 — Pipeline: Sentiment Analysis (No Training)

In [ ]:
from transformers import pipeline

sentiment = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english'
)

texts = [
    'I love using Hugging Face transformers!',
    'This movie was terrible and boring.',
    'The lecture was good but a bit long.',
    'I am not happy with the service.',
    'Amazing result, I am satisfied.'
]

results = sentiment(texts)
for t, r in zip(texts, results):
    print(f'Text: {t}')
    print(f" → Label: {r['label']} | Score: {r['score']:.4f}\n")

## Practice 2 — Tokenization (input_ids + attention_mask)
We will tokenize sentences using **BERT tokenizer** and inspect:
- tokens
- input_ids
- attention_mask

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

samples = [
    'Transformers are powerful for NLP.',
    'I poured water into the cup until it was full.',
    'Hugging Face makes models easy to use.'
]

encoded = tokenizer(
    samples,
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors='pt'
)

print('input_ids shape:', encoded['input_ids'].shape)
print('attention_mask shape:', encoded['attention_mask'].shape)

# Show tokens for first sentence
tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
print('\nTokens (sentence 1):')
print(tokens)

print('\ninput_ids (sentence 1):')
print(encoded['input_ids'][0].tolist())

print('\nattention_mask (sentence 1):')
print(encoded['attention_mask'][0].tolist())

## Practice 3 — BERT Hidden States (Embeddings)
We load **bert-base-uncased** and inspect:
- last_hidden_state shape
- [CLS] vector shape

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained('bert-base-uncased')

with torch.no_grad():
    outputs = model(**encoded)

last_hidden = outputs.last_hidden_state
print('Last hidden state shape:', last_hidden.shape)

# [CLS] token vector
cls_vector = last_hidden[:, 0, :]
print('CLS vector shape:', cls_vector.shape)

## Practice 4 — Zero-shot Classification (No Training)
Use BART MNLI to classify text into custom labels.

In [ ]:
from transformers import pipeline

zero_shot = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

text = 'This product is very good and I want to buy it again.'
labels = ['positive', 'negative', 'neutral']

out = zero_shot(text, labels)
print('Text:', out['sequence'])
for lbl, score in zip(out['labels'], out['scores']):
    print(f'{lbl}: {score:.4f}')

## Practice 5 — Fine-tuning DistilBERT on IMDb (Small Subset)
This is a **mini fine-tuning** demo for practice (fast).

Steps:
1) Load IMDb dataset
2) Tokenize
3) Fine-tune distilbert-base-uncased
4) Evaluate accuracy + F1

In [ ]:
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Load dataset
dataset = load_dataset('imdb')

# Small subset for practice
train_ds = dataset['train'].shuffle(seed=42).select(range(2000))
test_ds  = dataset['test'].shuffle(seed=42).select(range(1000))

model_name = 'distilbert-base-uncased'
tokenizer_ft = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer_ft(batch['text'], truncation=True, padding='max_length', max_length=128)

train_tok = train_ds.map(tokenize_fn, batched=True)
test_tok  = test_ds.map(tokenize_fn, batched=True)

# Remove raw text and set torch format
train_tok = train_tok.remove_columns(['text']).with_format('torch')
test_tok  = test_tok.remove_columns(['text']).with_format('torch')

model_ft = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'f1': f1.compute(predictions=preds, references=labels)['f1']
    }

args = TrainingArguments(
    output_dir='hf_practice_out',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to='none'
)

trainer = Trainer(
    model=model_ft,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer_ft,
    compute_metrics=compute_metrics
)

trainer.train()
metrics = trainer.evaluate()
print('Evaluation:', metrics)

## Practice 6 — Test the Fine-tuned Model

In [ ]:
from transformers import pipeline

# Create a pipeline from the fine-tuned model
clf = pipeline('sentiment-analysis', model=trainer.model, tokenizer=tokenizer_ft)

tests = [
    'This movie was fantastic!',
    'Worst experience ever, very bad.',
    'It was okay, not great not bad.'
]

print(clf(tests))

## ✅ Student Practice Tasks (Write Answers)
1) Replace the ticker sentences with your own and rerun pipeline.
2) Change `max_length` to 64 and compare tokenization output.
3) Try a different model in pipeline (e.g., `bert-base-uncased`).
4) Increase IMDb train subset to 5000 and check accuracy.
5) Write 5–8 lines: Why fine-tuning improves performance?